# 03 — Struggle test (Claim B)
Do teams that have been struggling in a game surge on a late-half drive, beyond what happens at other points? Definitions in `question-doc.md` §6, §7.

## History features
For each drive: what the same team did on its earlier drives in the same game.

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data")
drives = pd.read_parquet(DATA_DIR / "drives.parquet")

drives = drives.sort_values(["game_id", "posteam", "fixed_drive"]).reset_index(drop=True)
g = drives.groupby(["game_id", "posteam"])

drives["prior_n"] = g.cumcount()   # earlier drives by this team in this game
for col in ["is_td", "drive_epa", "first_downs", "three_and_out"]:
    # running total including this drive, minus this drive = earlier drives only
    drives[f"prior_{col}"] = g[col].cumsum().astype(float) - drives[col].astype(float)

drives[["game_id", "posteam", "fixed_drive", "prior_n", "prior_is_td", "prior_drive_epa"]].head(8)

,game_id,posteam,fixed_drive,prior_n,prior_is_td,prior_drive_epa
0,2016_01_BUF_BAL,BAL,1.0,0,0.0,0.000000
1,2016_01_BUF_BAL,BAL,3.0,1,0.0,-1.424516
2,2016_01_BUF_BAL,BAL,5.0,2,0.0,-5.283258
3,2016_01_BUF_BAL,BAL,7.0,3,0.0,-3.314834
4,2016_01_BUF_BAL,BAL,9.0,4,1.0,2.161842
5,2016_01_BUF_BAL,BAL,11.0,5,1.0,1.513576
6,2016_01_BUF_BAL,BAL,13.0,6,1.0,-1.020504
7,2016_01_BUF_BAL,BAL,15.0,7,1.0,-2.583396


## Struggled flags (§6)
Both definitions. Minimum 3 earlier drives in the same game; league averages are per season.

In [2]:
season_avg = drives.groupby("season").agg(avg_epa=("drive_epa", "mean"), avg_fd=("first_downs", "mean"))
drives = drives.merge(season_avg, on="season", how="left")

enough = drives["prior_n"] >= 3

# Definition 1 (scoring): no TDs, and below-average EPA per drive
drives["struggled_1"] = (enough & (drives["prior_is_td"] == 0)
                         & (drives["prior_drive_epa"] / drives["prior_n"] < drives["avg_epa"]))

# Definition 2 (ball movement): below-average first downs per drive, OR at least half three-and-outs
drives["struggled_2"] = (enough & ((drives["prior_first_downs"] / drives["prior_n"] < drives["avg_fd"])
                                   | (drives["prior_three_and_out"] / drives["prior_n"] >= 0.5)))

print("share with 3+ earlier drives:", round(enough.mean(), 3))
drives[["struggled_1", "struggled_2"]].mean().round(3)

share with 3+ earlier drives: 0.717


struggled_1    0.166
struggled_2    0.398
dtype: float64

In [3]:
e = drives[enough].copy()
fd_rate = e["prior_first_downs"] / e["prior_n"]
q25 = fd_rate.groupby(e["season"]).transform(lambda s: s.quantile(0.25))

pd.Series({
    "def2: below-avg first downs":       (fd_rate < e["avg_fd"]).mean(),
    "def2: >= half three-and-outs":      (e["prior_three_and_out"] / e["prior_n"] >= 0.5).mean(),
    "alt: bottom-quartile first downs":  (fd_rate < q25).mean(),
    "def1: no TD + below-avg EPA":       e["struggled_1"].mean(),
}).round(3)

def2: below-avg first downs         0.548
def2: >= half three-and-outs        0.112
alt: bottom-quartile first downs    0.247
def1: no TD + below-avg EPA         0.232
dtype: float64

In [1]:
# §6 Def 2 (v1.3): bottom-quartile first downs/drive for the season, OR >= half earlier drives three-and-out
n = drives["prior_n"].clip(lower=1)  # avoid divide-by-zero on a team's first drive
tao = drives["prior_three_and_out"] / n

drives["fd_rate"] = (drives["prior_first_downs"] / n).where(enough)
q25 = drives.loc[enough].groupby("season")["fd_rate"].quantile(0.25)  # cutoff per season, among drives with 3+ earlier drives
drives["fd_q25"] = drives["season"].map(q25)

drives["struggled_2_loose"] = enough & ((drives["fd_rate"] < drives["avg_fd"]) | (tao >= 0.5))  # original version
drives["struggled_2"] = enough & ((drives["fd_rate"] <= drives["fd_q25"]) | (tao >= 0.5))       # v1.3

sub = drives[enough]
print(sub[["struggled_1", "struggled_2", "struggled_2_loose"]].mean().round(3))
print("overlap, share of Def 2 that is also Def 1:", round((sub["struggled_1"] & sub["struggled_2"]).sum() / sub["struggled_2"].sum(), 3))

NameError: name 'drives' is not defined